# 12 . Spectral context on SG training, targeting the wiggle directly

Notebook 10/11 showed SG training improves M0/M1/M2 and, scored properly on the actual GI
wiggle (`score_sg_wiggle_loo.py`, PROGRESS.md 2026-09-10), makes the kinematic diagnostic
WORSE on average (dirty 0.664, frozen 0.641, finetune 0.564, fresh 0.522, mean resid r). Cause
is structural: the model gets one dirty channel in, one clean channel out, with zero
information about neighbouring velocity channels, so it cannot preserve a sub-channel
velocity centroid it never sees. Confirmed not a masking artifact (same PROGRESS entry).

Spectral context (`n_neighbors=k`) already exists and was proven on line emission
(`winner_k1`/`k2`, 2026-08-20/21): feed the model k channels either side, predict only the
centre channel. This applies the same lever to SG training for the first time.

**All three arms are `fresh` (random init).** No stored checkpoint has `in_channels` matching
k>0, so fine-tuning from `winner_aug` is not available here without first training a
matching-shape prior -- a fair fresh-vs-fresh comparison is what's available today. `k0` is a
same-day retrain of notebook 10's fresh arm as the control, not a reused checkpoint, so the
comparison isn't confounded by which Kaggle session ran it.

Same split as notebook 10 for direct comparability: train `{9015, 9019, 9032}`, val `9025`,
holdout `9074`.

## 0. Bootstrap

In [1]:
import os, sys, subprocess, glob

ON_KAGGLE = os.path.exists('/kaggle')
BRANCH = 'midterm-prep'
if ON_KAGGLE:
    REPO = '/kaggle/working/EXXA'; PKG = os.path.join(REPO, 'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',
                        'https://github.com/KrishanYadav333/EXXA.git',REPO], check=True)
    else:
        subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
        subprocess.run(['git','-C',REPO,'reset','--hard','origin/'+BRANCH], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps',
                    'pytorch-msssim','bettermoments'], check=True)
    os.chdir(os.path.join(PKG,'notebooks')); sys.path.insert(0, PKG)

    hits = [p for p in glob.glob('/kaggle/input/**/run_9*_rt_*', recursive=True) if os.path.isdir(p)]
    if not hits:
        raise FileNotFoundError('No run_9*_rt_* folders under /kaggle/input -- attach exxa-sg-synth-pairs.')
    DATA_DIR = os.path.dirname(hits[0])
else:
    if os.path.basename(os.getcwd()) != 'notebooks' and os.path.isdir('notebooks'):
        os.chdir('notebooks')
    sys.path.insert(0, os.path.abspath('..'))
    DATA_DIR = '../self-gravitating cube and dirty cube/sg_synth'

print('DATA_DIR:', DATA_DIR)

Cloning into '/kaggle/working/EXXA'...
Updating files: 100% (5456/5456), done.


DATA_DIR: /kaggle/input/datasets/krishanyadav333/exxa-sg-synth-pairs/kaggle-sg-training-dataset


## 0b. Pull latest `src/`

In [2]:
if ON_KAGGLE:
    subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard','FETCH_HEAD'], check=True)
    print(subprocess.run(['git','-C',REPO,'log','--oneline','-1'], capture_output=True, text=True).stdout)
    import importlib, src; importlib.reload(src)

From https://github.com/KrishanYadav333/EXXA
 * branch            midterm-prep -> FETCH_HEAD


HEAD is now at 07f047f docs: day-by-day checklist to Nov 2
07f047f docs: day-by-day checklist to Nov 2



## 1. Config

In [3]:
import time, math, json
import numpy as np
import torch
import torch.nn.functional as Fn
from torch.utils.data import DataLoader
from astropy.io import fits

from src.data.cube_split import list_cubes
from src.data.fits_cube_dataset import FITSChannelDataset
from src.training.sweep import train_unet, val_metrics
from src.training.architectures import build_model
from src.models.unet import UNet
from src.evaluation.moment_maps import generate_moment_maps, signal_mask, moment_improvement
from src.evaluation.gi_wiggle import quadratic_moment1, fit_keplerian, wiggle_residual, wiggle_amplitude

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

TARGET_SIZE = 256
N_SAMPLES = 120
NW = 2 if torch.cuda.is_available() else 0

WINNER = dict(base_channels=48, channel_multipliers=(1, 2, 4, 8),
              lr=8.196504330730313e-4, alpha=0.8877681051398497,
              sched_patience=8, batch_size=8)
KS = [0, 1, 2]          # spectral-context radius: k0 is the control

TRAIN_RUNS = ['run_9015_00370_rt_00', 'run_9019_00019_rt_00', 'run_9032_00020_rt_00']
VAL_RUN = 'run_9025_00370_rt_00'
HOLD_RUN = 'run_9074_00025_rt_00'
HOLD_TRUE_INCL = 20.0   # this disk's .para: 1.0 Msun, 20 deg, 175.178 pc

CKPT_DIR = '../results/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'{device} | seed {SEED} | k in {KS}')

cuda | seed 42 | k in [0, 1, 2]


## 2. Data

In [4]:
all_cubes = {c['folder']: c for c in list_cubes(DATA_DIR)}
train_cubes = [all_cubes[r] for r in TRAIN_RUNS]
val_cubes = [all_cubes[VAL_RUN]]
hold = all_cubes[HOLD_RUN]
print('train:', [c['folder'] for c in train_cubes])
print('val:  ', [c['folder'] for c in val_cubes])
print('holdout:', hold['folder'])

datasets = {}
for k in KS:
    _kw = dict(n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED,
               subtract_continuum=False, n_neighbors=k, stack_target=False, verbose=False)
    tr = FITSChannelDataset(train_cubes, **_kw)
    va = FITSChannelDataset(val_cubes, **_kw)
    d, c = tr[0]
    print(f'k={k}: dirty {tuple(d.shape)} clean {tuple(c.shape)}  train={len(tr)} val={len(va)}')
    assert d.shape[0] == 2 * k + 1 and c.shape[0] == 1
    datasets[k] = (tr, va)

train: ['run_9015_00370_rt_00', 'run_9019_00019_rt_00', 'run_9032_00020_rt_00']
val:   ['run_9025_00370_rt_00']
holdout: run_9074_00025_rt_00
k=0: dirty (1, 256, 256) clean (1, 256, 256)  train=360 val=120
k=1: dirty (3, 256, 256) clean (1, 256, 256)  train=360 val=120
k=2: dirty (5, 256, 256) clean (1, 256, 256)  train=360 val=120


## 3. Train (all `fresh`, random init, since no matching-shape checkpoint exists for k>0)

In [5]:
results = {}
for k in KS:
    name = f'sg_k{k}_fresh'
    tr, va = datasets[k]
    print(f"\n{'='*70}\nk={k}  in_channels={2*k+1}\n{'='*70}")
    t0 = time.time()
    out = train_unet(tr, va, device, n_neighbors=k, out_channels=1,
                     min_epochs=10, max_epochs=60, patience=8,
                     num_workers=NW, seed=SEED,
                     ckpt_path=os.path.join(CKPT_DIR, f'{name}.pth'),
                     verbose=False, **WINNER)
    out['minutes'] = (time.time() - t0) / 60
    results[k] = out
    print(f"  PSNR {out['psnr']:.3f} dB, SSIM {out['ssim']:.5f}, best epoch {out['best_epoch']}, "
          f"{out['minutes']:.1f} min")


k=0  in_channels=1
  PSNR 30.053 dB, SSIM 0.98158, best epoch 46, 29.0 min

k=1  in_channels=3
  PSNR 32.828 dB, SSIM 0.98723, best epoch 33, 22.3 min

k=2  in_channels=5
  PSNR 33.576 dB, SSIM 0.98836, best epoch 29, 20.4 min


## 4. Score on the holdout: moments AND the wiggle

In [6]:
def denoise_k(state_dict, cube, k):
    net = build_model('unet', base_channels=WINNER['base_channels'],
                      channel_multipliers=WINNER['channel_multipliers'],
                      use_beam=False, n_neighbors=k, out_channels=1, latent_dim=128).to(device)
    miss, unexp = net.load_state_dict(state_dict, strict=False)
    assert not miss and not unexp
    net.eval()
    C, H, W = cube.shape
    los = cube.reshape(C, -1).min(axis=1); his = cube.reshape(C, -1).max(axis=1)
    rng = his - los
    out = np.empty_like(cube, dtype=np.float32)
    with torch.no_grad():
        for s in range(0, C, 8):
            idx = np.arange(s, min(s + 8, C))
            if k > 0:
                nb = np.clip(idx[:, None] + np.arange(-k, k + 1)[None, :], 0, C - 1)
                lo, hi = los[idx][:, None, None, None], his[idx][:, None, None, None]
                rr = hi - lo
                blk = np.where(rr > 0, (cube[nb].astype(np.float64) - lo) / np.where(rr > 0, rr, 1), 0.0)
                t = torch.from_numpy(blk).float().to(device)
            else:
                lo, hi = los[idx], his[idx]
                den = np.where((hi - lo) > 0, hi - lo, 1)[:, None, None]
                blk = (cube[idx].astype(np.float64) - lo[:, None, None]) / den
                t = torch.from_numpy(blk)[:, None].float().to(device)
            t = Fn.interpolate(t, (TARGET_SIZE, TARGET_SIZE), mode='bilinear', align_corners=False)
            p = net(t, torch.zeros(t.size(0), dtype=torch.long, device=device), None)
            b = Fn.interpolate(p, (H, W), mode='bilinear', align_corners=False)[:, 0].cpu().numpy()
            for j, ch in enumerate(idx):
                out[ch] = b[j] * rng[ch] + los[ch] if rng[ch] > 0 else los[ch]
    return out


with fits.open(hold['clean'], memmap=True) as h:
    hdr = h[0].header; clean_cube = h[0].data[:]
with fits.open(hold['dirty'], memmap=True) as h:
    dirty_cube = h[0].data[:]
velax = (hdr['CRVAL3'] + (np.arange(clean_cube.shape[0]) + 1 - hdr['CRPIX3']) * hdr['CDELT3']) * 1000.0
au_per_px = abs(hdr['CDELT1']) * 3600.0 * float(hdr['DIST_PC'])
print(f'holdout {clean_cube.shape}, dv {hdr["CDELT3"]:.4f} km/s')

m_clean = generate_moment_maps('', data_velax=(clean_cube.astype(np.float64), velax))
m_dirty = generate_moment_maps('', data_velax=(dirty_cube.astype(np.float64), velax))
mask = signal_mask(m_clean[0], frac=0.05)

m1_clean, _ = quadratic_moment1(clean_cube.astype(np.float64), velax)
m1_clean = m1_clean / 1000.0
geom = fit_keplerian(m1_clean, mask, au_per_px, fix_incl_deg=HOLD_TRUE_INCL)
ref_resid = wiggle_residual(m1_clean, geom)
print(f'geometry (incl fixed {HOLD_TRUE_INCL}): mstar={geom["mstar_msun"]:.3f} at_bound={geom["mstar_at_bound"]}')

scores = {}
for k in KS:
    ck = torch.load(os.path.join(CKPT_DIR, f'sg_k{k}_fresh.pth'), map_location='cpu', weights_only=False)
    den = denoise_k(ck['model_state_dict'], dirty_cube, k)
    m_den = generate_moment_maps('', data_velax=(den.astype(np.float64), velax))
    imp = moment_improvement(m_clean, m_dirty, m_den)

    v0, _ = quadratic_moment1(den.astype(np.float64), velax)
    m1_den = v0 / 1000.0
    resid = wiggle_residual(m1_den, geom)
    ok = np.isfinite(ref_resid[mask]) & np.isfinite(resid[mask])
    resid_r = float(np.corrcoef(ref_resid[mask][ok], resid[mask][ok])[0, 1]) if ok.sum() > 10 else float('nan')

    scores[k] = dict(moments=imp, resid_r=resid_r, psnr=results[k]['psnr'], ssim=results[k]['ssim'])
    print(f"k={k}  PSNR {results[k]['psnr']:6.3f}  M0 {imp['M0']:+7.1f}  M1 {imp['M1']:+7.1f}  "
          f"M2 {imp['M2']:+7.1f}  wiggle resid_r {resid_r:.4f}")

holdout (201, 301, 301), dv 0.1000 km/s
geometry (incl fixed 20.0): mstar=0.676 at_bound=False
k=0  PSNR 30.053  M0   -26.5  M1    +4.2  M2   -44.4  wiggle resid_r 0.3660
k=1  PSNR 32.828  M0   +29.2  M1   +31.0  M2   +62.5  wiggle resid_r 0.5904
k=2  PSNR 33.576  M0   +61.4  M1   +31.4  M2   +67.6  wiggle resid_r 0.5063


## 5. Results

In [7]:
print(f"{'k':>3} {'PSNR':>8} {'M0':>8} {'M1':>8} {'M2':>8} {'wiggle resid_r':>15}")
for k in KS:
    s = scores[k]
    print(f"{k:3d} {s['psnr']:8.3f} {s['moments']['M0']:8.1f} {s['moments']['M1']:8.1f} "
          f"{s['moments']['M2']:8.1f} {s['resid_r']:15.4f}")

print()
print('reference from notebook 11 fold 4 (same holdout, k=0, different training run):')
print('  frozen resid_r 0.487, finetune (from winner_aug) 0.465, fresh 0.278')

os.makedirs('../results/self-gravitating', exist_ok=True)
with open('../results/self-gravitating/sg_spectral_context_wiggle.json', 'w') as f:
    json.dump({str(k): scores[k] for k in KS}, f, indent=2, default=str)
print('\nsaved -> results/self-gravitating/sg_spectral_context_wiggle.json')

  k     PSNR       M0       M1       M2  wiggle resid_r
  0   30.053    -26.5      4.2    -44.4          0.3660
  1   32.828     29.2     31.0     62.5          0.5904
  2   33.576     61.4     31.4     67.6          0.5063

reference from notebook 11 fold 4 (same holdout, k=0, different training run):
  frozen resid_r 0.487, finetune (from winner_aug) 0.465, fresh 0.278

saved -> results/self-gravitating/sg_spectral_context_wiggle.json
